# Avocado Prices — Process Phase

Drop a leftover index column, standardize column names, and — the central task — classify the
`region` column into its true 3-level hierarchy (national / major region / city) so later
aggregation never silently double-counts volume. Source: `data/raw/avocado.csv` (not committed —
see `data/raw/README.md`). Output: `data/processed/avocado_clean.parquet`.

## Step 1 — Load raw data

In [1]:
import pandas as pd
import os

RAW = "../data/raw"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(RAW, "avocado.csv"), parse_dates=["Date"])
print(f"Loaded: {df.shape}")

Loaded: (18249, 14)

## Step 2 — Drop the leftover index column, standardize column names

In [2]:
df = df.drop(columns=["Unnamed: 0"])
df = df.rename(columns={
    "Date": "date", "AveragePrice": "average_price", "Total Volume": "total_volume",
    "4046": "volume_small", "4225": "volume_medium", "4770": "volume_large",
    "Total Bags": "total_bags", "Small Bags": "small_bags", "Large Bags": "large_bags",
    "XLarge Bags": "xlarge_bags", "type": "type", "year": "year", "region": "region",
})
print(f"Columns: {list(df.columns)}")

Columns: ['date', 'average_price', 'total_volume', 'volume_small', 'volume_medium', 'volume_large', 'total_bags', 'small_bags', 'large_bags', 'xlarge_bags', 'type', 'year', 'region']

## Step 3 — Classify `region` into its true hierarchy

Verified in the Prepare phase: the 8 "major regions" sum to exactly `TotalUS`; the 45 "city"
entries are a separate, finer-grained (but incomplete) breakdown. Tagging each row with its tier
prevents any later aggregation from mixing levels and overcounting.

In [3]:
major_regions = {"California", "GreatLakes", "Midsouth", "Northeast", "Plains",
                  "SouthCentral", "Southeast", "West"}

def tier(region):
    if region == "TotalUS":
        return "national"
    elif region in major_regions:
        return "major_region"
    else:
        return "city"

df["region_tier"] = df["region"].apply(tier)
print(df["region_tier"].value_counts())

region_tier
city            15207
major_region     2704
national          338
Name: count, dtype: int64

In [4]:
# Re-verify the hierarchy holds after tagging
check = df[(df["date"] == "2015-12-27") & (df["type"] == "conventional")]
total_us = check.loc[check["region"] == "TotalUS", "total_volume"].sum()
majors_sum = check.loc[check["region_tier"] == "major_region", "total_volume"].sum()
print(f"TotalUS={total_us:,.0f}, sum of majors={majors_sum:,.0f}, diff={total_us-majors_sum:,.2f}")

TotalUS=27,297,984, sum of majors=27,297,984, diff=0.03

## Step 4 — Check the weekly date cadence

In [5]:
irregular_groups = df.groupby(["region", "type"])["date"].apply(
    lambda s: (s.sort_values().diff().dropna() != pd.Timedelta(days=7)).sum()
)
print(f"Groups with at least one non-7-day gap: {(irregular_groups > 0).sum()} / {len(irregular_groups)}")
print(f"Total irregular gaps across all groups: {irregular_groups.sum()}")

Groups with at least one non-7-day gap: 1 / 108
Total irregular gaps across all groups: 2

A single region/type series has 2 non-weekly gaps out of 108 series — minor, left as-is (no data was fabricated to fill it).

## Step 5 — Check for price/volume outliers

In [6]:
print(df["average_price"].describe())
print(f"Rows with average_price <= 0: {(df['average_price'] <= 0).sum()}")
print(f"Rows with total_volume < 0: {(df['total_volume'] < 0).sum()}")

count    18249.000000
mean         1.405978
std          0.402677
min          0.440000
25%          1.100000
50%          1.370000
75%          1.660000
max          3.250000
Name: average_price, dtype: float64
Rows with average_price <= 0: 0
Rows with total_volume < 0: 0

No invalid prices or volumes — no rows need to be dropped for data-quality reasons.

## Step 6 — Save the processed dataset

In [7]:
df.to_parquet(os.path.join(OUT_DIR, "avocado_clean.parquet"), index=False)
size_kb = os.path.getsize(os.path.join(OUT_DIR, "avocado_clean.parquet")) / 1024
print(f"Saved avocado_clean.parquet ({size_kb:.1f} KB), {df.shape}")

Saved avocado_clean.parquet (952.8 KB), (18249, 14)

## Verification

The DuckDB SQL pipeline ([`sql/01_process_data.sql`](../sql/01_process_data.sql)) reproduces the
same row count (18,249), region count (54), and hierarchy check (majors sum to TotalUS within
floating-point rounding).

## Summary

| Step | Result |
|---|---|
| Dropped leftover index column | 14 → 13 real columns |
| Region hierarchy classification | 338 national, 2,704 major-region, 15,207 city rows |
| Hierarchy verified | 8 majors sum to TotalUS exactly (both engines) |
| Date cadence | 106/108 series perfectly weekly; 2 minor gaps in 1 series |
| Price/volume outliers | None found |